# Setup

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [2]:
print(len(documents))

72


In [3]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [11]:
import evaluation_utils
from openai import OpenAI
from pydantic import BaseModel
import json
import pandas as pd

In [5]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.environ["GEMINI_API_KEY"]

print("working")

working


In [6]:

client = OpenAI(
    api_key=os.environ.get("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

Question pydantic model

In [7]:
class Questions(BaseModel):
    questions: list[str]

# Question 1

In [8]:
page1 = next(d for d in documents if d["filename"] == "01-agentic-rag/lessons/01-intro.md" )
page2 = next(d for d in documents if d["filename"] == "01-agentic-rag/lessons/02-environment.md" )
page3 = next(d for d in documents if d["filename"] == "01-agentic-rag/lessons/03-rag.md" )
pages = [page1, page2, page3]

In [9]:
results = []
usages = []
for page in pages:
    user_prompt = json.dumps({
        "filename" : page["filename"],
        "content" : page["content"]
    })
    
    questions, usage = evaluation_utils.llm_structured(
        client,
        instructions=data_gen_instructions,
        user_prompt=user_prompt,
        output_type=Questions,
        model="gemini-2.5-flash"
    )

    results.append(questions)
    usages.append(usage)

    print(page["filename"])
    print(questions)
    print(usage)
    print("="*7)

01-agentic-rag/lessons/01-intro.md
questions=["What's the basic idea behind a Large Language Model (LLM)?", 'What are the main problems or shortcomings of using LLMs without any extra help?', 'How does RAG technology help overcome these limitations of LLMs?', 'What kind of practical system will we be constructing as our main project in this module?', 'Will we be using complex frameworks to build our RAG system, or taking a different approach?']
CompletionUsage(completion_tokens=90, prompt_tokens=1064, total_tokens=2102, completion_tokens_details=None, prompt_tokens_details=None)
01-agentic-rag/lessons/02-environment.md
questions=['What Python version and external accounts are required to get started with this module?', 'When creating a new project from scratch, what is the recommended tool for managing Python packages, and how do I initialize a new project folder with it?', "Could you explain the purpose of the main Python libraries we're adding, such as `minsearch` and `python-dotenv`

In [10]:
avg_input_tokens = sum(u.prompt_tokens for u in usages) / len(usages)
avg_input_tokens

1450.6666666666667

**Answer**: 1400

# Question 2

In [40]:
ground_truth = pd.read_csv("ground-truth.csv")
ground_truth = ground_truth.to_dict(orient="records")

In [13]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

In [41]:
from minsearch import Index

idx = Index(text_fields=["content"], keyword_fields=["filename"])
idx.fit(chunks)

def text_search(query, num_results=5):
    return idx.search(
        query=query,
        num_results=num_results,
    )



In [42]:
q = ground_truth[0]["question"]
q

"What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?"

In [47]:
text_search_result = text_search(q)
print(text_search_result [0]["filename"])

01-agentic-rag/lessons/03-rag.md


**Answer:** 01-agentic-rag/lessons/03-rag.md

# Question 3

In [44]:
from embedder import Embedder

embedder = Embedder()

chunks_contents = [c["content"] for c in chunks]

embedded_chunks = embedder.encode_batch(chunks_contents)

In [45]:
from minsearch import VectorSearch
from sklearn.metrics.pairwise import cosine_similarity

def fixed_search(self, query_vector, filter_dict=None, num_results=10, output_ids=False):
    if len(self.docs) == 0 or self.vectors is None:
        return []
    query_vector_2d = query_vector.reshape(1, -1)
    scores = cosine_similarity(query_vector_2d, self.vectors).flatten()
    top_indices = scores.argsort()[::-1][:num_results]
    return [self.docs[i] for i in top_indices]

VectorSearch.search = fixed_search

vs_index = VectorSearch()          
vs_index.fit(embedded_chunks, chunks)

def vector_search(query, num_results=5):     
    query_vector = embedder.encode(query)
    return vs_index.search(query_vector, num_results=num_results)

In [46]:
results = vector_search(q)
print(results[0]["filename"])

01-agentic-rag/lessons/01-intro.md


**Answer:** 01-agentic-rag/lessons/01-intro.md

# Question 4

In [51]:
def compute_relevance(search_function, q):
    results = search_function(q["question"])
    return [d["filename"] == q["filename"] for d in results]

In [52]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [53]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [54]:
def evaluate(search_function, ground_truth):
    relevance_total = [compute_relevance(search_function, q) for q in ground_truth]
    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [58]:
text_eval_results = evaluate( text_search, ground_truth=ground_truth )

In [59]:
text_eval_results["hit_rate"]

0.7583333333333333

**Answer** : 0.76

# Question 5

In [60]:
vector_eval_results = evaluate(vector_search, ground_truth=ground_truth)

In [61]:
vector_eval_results["mrr"]

0.5486111111111112

**Answer:** 0.55

# Question 6

In [62]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [63]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [70]:
for k_value in [1, 50, 100, 200]:
    search_fn = lambda q, k=k_value: hybrid_search(q, k=k)
    results = evaluate(search_fn, ground_truth)
    print(f"{k_value} --> {results['mrr']}")
    print("----------------------")

1 --> 0.6481944444444449
----------------------
50 --> 0.637916666666667
----------------------
100 --> 0.637916666666667
----------------------
200 --> 0.637916666666667
----------------------


**Answer:** 1